In [0]:
# COMMAND ----------
# ── CELL 1: Install & Imports ──────────────────────────────────────────────
 
# Faker is not pre-installed on all DBR versions — install if needed
%pip install faker --quiet
 
# COMMAND ----------
 
import random
import uuid
from datetime import datetime, timedelta, date
 
from faker import Faker
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType,
    BooleanType, DateType, TimestampType
)
 
fake = Faker()
Faker.seed(42)
random.seed(42)
 
spark = SparkSession.builder.getOrCreate()
 
print("✅ Imports complete")

In [0]:
# Unity Catalog target
CATALOG = "icebase"
SCHEMA  = "bronze"
 
# Season configuration
SEASON_START_DATE = date(2024, 10, 4)   # Opening night
JERSEY_NIGHT_GAME = 77                  # Mack Tateson #14 retirement game
TOTAL_GAMES       = 82
 
# Fan population
N_CUSTOMERS_BASE  = 5_000              # Seed fan base
N_SEASON_HOLDERS  = 420                # ~8.4% of base are season ticket holders
 
# Arena config
ARENA_NAME        = "Gem State Arena"
ARENA_CAPACITY    = 6_200
 
# Reproducibility
RANDOM_SEED = 42
 
print(f"✅ Config loaded | Catalog: {CATALOG}.{SCHEMA} | Arena Capacity: {ARENA_CAPACITY:,}")

In [0]:
# COMMAND ----------
# ── CELL 3: Season Phase Weights ──────────────────────────────────────────
#
# This is the NARRATIVE ENGINE of IceBase.
# Every data generation function references these weights by game number
# to encode the business story into the data itself.
#
# Keys per phase:
#   fill_rate       : tuple (min, max) — fraction of arena sold
#   price_mult      : float — multiplier on base ticket price ($55)
#   promo_rate      : float — fraction of tickets using a promo code
#   new_fan_weight  : float — relative weight of new customer acquisition
#   churn_inject    : str   — which segments face churn pressure this phase
#   notes           : str   — business narrative context
 
PHASE_WEIGHTS = {
    "hot_start": {
        "games":        range(1, 23),
        "fill_rate":    (0.88, 0.95),
        "price_mult":   1.15,
        "promo_rate":   0.08,
        "new_fan_weight": 1.40,
        "churn_inject": None,
        "notes": "Team opens hot. Organic demand. Walk-ups and social signups surge."
    },
    "slump": {
        "games":        range(23, 53),
        "fill_rate":    (0.58, 0.67),
        "price_mult":   0.82,
        "promo_rate":   0.42,
        "new_fan_weight": 0.80,
        "churn_inject": "casual,social",
        "notes": "Six-game skid. Star injured. Marketing floods promos to prop attendance."
    },
    "late_push": {
        "games":        range(53, 73),
        "fill_rate":    (0.78, 0.84),
        "price_mult":   1.02,
        "promo_rate":   0.18,
        "new_fan_weight": 1.00,
        "churn_inject": "promo_hunter",
        "notes": "Team gets healthy. Playoff race heats up. Promo-conditioned fans slow to return."
    },
    "jersey_night": {
        "games":        range(77, 78),
        "fill_rate":    (1.00, 1.00),
        "price_mult":   2.40,
        "promo_rate":   0.04,
        "new_fan_weight": 2.80,
        "churn_inject": None,
        "notes": "Mack Tateson #14 retired. Sellout. Lapsed fans reactivate. Massive new acquisition."
    },
    "post_jersey": {
        "games":        range(78, 83),
        "fill_rate":    (0.82, 0.88),
        "price_mult":   1.10,
        "promo_rate":   0.10,
        "new_fan_weight": 1.15,
        "churn_inject": None,
        "notes": "Momentum from Jersey Night carries into playoff push."
    },
}
 
# Games 73–76 bridge between late push and jersey night (normal operations)
BRIDGE_WEIGHT = {
    "fill_rate":    (0.76, 0.82),
    "price_mult":   1.00,
    "promo_rate":   0.15,
    "new_fan_weight": 0.95,
    "churn_inject": None,
}
 
def get_phase(game_number: int) -> dict:
    """Return the phase weight dict for a given game number."""
    for phase_name, phase in PHASE_WEIGHTS.items():
        if game_number in phase["games"]:
            return phase
    return {**BRIDGE_WEIGHT, "notes": "Bridge game — normal operations", "games": []}
 
print("✅ Phase weights configured")
for name, p in PHASE_WEIGHTS.items():
    print(f"   {name:<15} | Games {list(p['games'])[0]:>2}–{list(p['games'])[-1]:>2} "
          f"| Fill: {p['fill_rate']} | Promo: {p['promo_rate']:.0%} | {p['notes'][:55]}")

In [0]:
# COMMAND ----------
# ── CELL 4: Unity Catalog Setup ───────────────────────────────────────────
 
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")
 
print(f"✅ Unity Catalog ready: {CATALOG}.(bronze | silver | gold)")

In [0]:
# COMMAND ----------
# ── CELL 5: Generate raw_events (82-game schedule) ─────────────────────────
#
# BUSINESS CONTEXT:
#   Full 82-game regular season for the Idaho Mashers.
#   Game outcomes are weighted to match the narrative arc:
#     - Hot start: high win rate
#     - Slump:     low win rate, star player injured game 23
#     - Late push: recovering win rate
#     - Jersey Night: win (of course)
 
OPPONENTS = [
    "Boise Bears", "Portland Timberwolves", "Salt Lake Falcons",
    "Reno Vipers", "Spokane Thunderbirds", "Eugene Otters",
    "Tacoma Fury", "Bend Ice Cats", "Missoula Raiders", "Provo Predators"
]
 
# Win probability by phase — encodes team performance narrative
WIN_PROB_BY_PHASE = {
    "hot_start":    0.72,
    "slump":        0.31,
    "late_push":    0.61,
    "jersey_night": 1.00,   # They win on jersey night. Obviously.
    "post_jersey":  0.65,
    "bridge":       0.50,
}
 
def get_win_prob(game_number: int) -> float:
    for phase_name, phase in PHASE_WEIGHTS.items():
        if game_number in phase["games"]:
            return WIN_PROB_BY_PHASE.get(phase_name, 0.50)
    return WIN_PROB_BY_PHASE["bridge"]
 
events = []
game_date = SEASON_START_DATE
 
for game_num in range(1, TOTAL_GAMES + 1):
    # Spread games ~3.5 days apart across the season
    game_date += timedelta(days=random.randint(2, 5))
    
    phase   = get_phase(game_num)
    is_home = random.random() > 0.45   # ~55% home games
    opponent = random.choice(OPPONENTS)
    
    # Score simulation
    win     = random.random() < get_win_prob(game_num)
    home_g  = random.randint(3, 6) if win else random.randint(1, 3)
    away_g  = random.randint(1, 3) if win else random.randint(3, 6)
    
    # Fill rate from phase weights
    fill_min, fill_max = phase["fill_rate"]
    fill_rate = round(random.uniform(fill_min, fill_max), 3)
    attendance = int(ARENA_CAPACITY * fill_rate)
 
    # Special flags
    is_jersey_night      = (game_num == JERSEY_NIGHT_GAME)
    is_star_injured_game = (game_num == 23)   # Injury announcement game
 
    events.append(Row(
        game_id             = f"MASH-2024-{game_num:03d}",
        game_number         = game_num,
        game_date           = game_date,
        opponent            = opponent,
        is_home_game        = is_home,
        arena               = ARENA_NAME if is_home else f"{opponent.split()[0]} Arena",
        home_score          = home_g,
        away_score          = away_g,
        result              = "W" if win else "L",
        attendance          = attendance,
        arena_fill_pct      = fill_rate,
        season_type         = "regular",
        season_phase        = next(
            (k for k, v in PHASE_WEIGHTS.items() if game_num in v["games"]), "bridge"
        ),
        is_jersey_night     = is_jersey_night,
        is_star_injury_game = is_star_injured_game,
        _ingested_at        = datetime.now(),
        _source             = "seed_generator_v1",
    ))
 
events_df = spark.createDataFrame(events)
events_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.bronze.raw_events")
 
print(f"✅ raw_events written | {events_df.count()} games")
print(f"   Wins: {sum(1 for e in events if e.result == 'W')} | "
      f"Losses: {sum(1 for e in events if e.result == 'L')}")
print(f"   Jersey Night (Game {JERSEY_NIGHT_GAME}): "
      f"{next(e for e in events if e.is_jersey_night).game_date}")

In [0]:
# COMMAND ----------
# ── CELL 6: Generate raw_customers (5,000 fan profiles) ──────────────────
#
# BUSINESS CONTEXT:
#   Fan base is seeded with realistic acquisition channel distribution.
#   Season holders are flagged and given earlier signup dates (loyal base).
#   A cohort of "deeply lapsed" fans is created for the Jersey Night story —
#   these fans haven't attended in 3+ years but will reactivate on game 77.
 
ACQUISITION_CHANNELS = [
    ("season_package",  0.09),   # Direct season holder
    ("walk_up",         0.18),   # Game day walk-up
    ("social_media",    0.22),   # Organic social / Instagram
    ("friend_referral", 0.20),   # Word of mouth
    ("email_campaign",  0.14),   # Marketing campaign
    ("promo_code",      0.11),   # Discount-driven
    ("corporate",       0.06),   # B2B / suite holders
]
channels, channel_weights = zip(*ACQUISITION_CHANNELS)
 
ID_SEGMENTS = [
    "season_core",      # ~8%  — loyal, full-price, high value
    "casual_fan",       # ~35% — attends 2–5 games, price sensitive
    "promo_hunter",     # ~20% — promo-driven, low organic spend
    "high_value_new",   # ~10% — acquired during hot start, strong potential
    "lapsed",           # ~22% — hasn't purchased in 45+ days
    "deeply_lapsed",    # ~5%  — 1000+ days since last purchase (Jersey Night cohort)
]
SEGMENT_WEIGHTS = [0.08, 0.35, 0.20, 0.10, 0.22, 0.05]
 
IDAHO_ZIPS = [
    "83401", "83402", "83404",  # Idaho Falls
    "83301", "83302",           # Twin Falls
    "83201", "83202",           # Pocatello
    "83647", "83651",           # Mountain Home / Nampa
    "83854", "83858",           # Post Falls / Rathdrum
]
 
customers = []
season_holder_ids = set()
 
for i in range(N_CUSTOMERS_BASE):
    cid        = str(uuid.uuid4())
    segment    = random.choices(ID_SEGMENTS, weights=SEGMENT_WEIGHTS)[0]
    channel    = random.choices(channels, weights=channel_weights)[0]
    is_holder  = (segment == "season_core") or (channel == "season_package")
 
    # Signup date — season holders joined earlier
    if is_holder:
        signup_days_ago = random.randint(300, 900)
    elif segment == "deeply_lapsed":
        signup_days_ago = random.randint(1200, 2500)
    elif segment == "lapsed":
        signup_days_ago = random.randint(60, 400)
    else:
        signup_days_ago = random.randint(10, 300)
 
    signup_date = date.today() - timedelta(days=signup_days_ago)
 
    # Email — small intentional dirtiness for Silver to clean
    first = fake.first_name()
    last  = fake.last_name()
    email_formats = [
        f"{first.lower()}.{last.lower()}@{fake.free_email_domain()}",
        f"{first.lower()}{random.randint(1,99)}@{fake.free_email_domain()}",
        f"{first[0].lower()}{last.lower()}@{fake.free_email_domain()}",
        f"{first.lower()}_{last.lower()}@{fake.free_email_domain()}",
    ]
    email = random.choice(email_formats)
 
    # Dirty name field — occasional ALL CAPS or extra whitespace (realistic)
    name_variants = [
        f"{first} {last}",
        f"{first.upper()} {last.upper()}",
        f"  {first} {last}  ",
        f"{first} {last}",
    ]
    full_name = random.choices(name_variants, weights=[0.80, 0.08, 0.07, 0.05])[0]
 
    if is_holder:
        season_holder_ids.add(cid)
 
    customers.append(Row(
        customer_id          = cid,
        full_name            = full_name,
        email                = email,
        phone                = fake.phone_number(),
        zip_code             = random.choice(IDAHO_ZIPS),
        state                = "ID",
        acquisition_channel  = channel,
        initial_segment      = segment,
        is_season_holder     = is_holder,
        signup_date          = signup_date,
        is_jersey_night_acq  = False,    # Will be set True for game-77 new fans
        is_deeply_lapsed     = (segment == "deeply_lapsed"),
        _ingested_at         = datetime.now(),
        _source              = "seed_generator_v1",
    ))
 
customers_df = spark.createDataFrame(customers)
customers_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.bronze.raw_customers")
 
print(f"✅ raw_customers written | {customers_df.count():,} fans")
print(f"   Season holders:    {len(season_holder_ids):,}")
print(f"   Deeply lapsed:     {sum(1 for c in customers if c.is_deeply_lapsed):,}")
seg_counts = {}
for c in customers:
    seg_counts[c.initial_segment] = seg_counts.get(c.initial_segment, 0) + 1
for seg, cnt in sorted(seg_counts.items(), key=lambda x: -x[1]):
    print(f"   {seg:<20}: {cnt:>5} ({cnt/N_CUSTOMERS_BASE:.1%})")

In [0]:
# COMMAND ----------
# ── CELL 7: Generate raw_tickets (transaction history) ────────────────────
#
# BUSINESS CONTEXT:
#   Ticket purchase history is the heart of the data model.
#   Volume, price, and promo usage are all weighted per season phase.
#
#   Special cases built in:
#     - Season holders: pre-purchase ALL home games (realistic)
#     - Slump games:    heavy promo attach, lower prices, fewer unique buyers
#     - Jersey Night:   premium pricing, sellout, lapsed fan reactivation
#     - Walk-up/social cohort: buys once during hot start, mostly disappears
#
#   Churn story encoded by simply NOT generating future tickets for
#   lapsed and casual segments after game 35 (slump deepens).
 
BASE_TICKET_PRICE = 55.00
 
SECTION_TIERS = {
    "rinkside":  {"price_add": 65.0, "pct": 0.05},
    "lower_bowl": {"price_add": 25.0, "pct": 0.20},
    "mid_bowl":  {"price_add": 10.0,  "pct": 0.35},
    "upper_bowl": {"price_add": 0.0,  "pct": 0.30},
    "standing":  {"price_add": -10.0, "pct": 0.10},
}
tier_names    = list(SECTION_TIERS.keys())
tier_weights  = [v["pct"] for v in SECTION_TIERS.values()]
tier_price_add = [v["price_add"] for v in SECTION_TIERS.values()]
 
customer_list = customers   # already built in cell 6
event_list    = events      # already built in cell 5
home_events   = [e for e in event_list if e.is_home_game]
 
tickets      = []
ticket_count = 0
 
# ── Helper: build one ticket row ──────────────────────────────────────────
def make_ticket(cid, game, phase, is_promo=False, is_jersey_night=False,
                is_lapsed_reactivation=False, is_new_jersey_acq=False):
    tier_idx  = random.choices(range(len(tier_names)), weights=tier_weights)[0]
    tier      = tier_names[tier_idx]
    price_add = tier_price_add[tier_idx]
    base      = BASE_TICKET_PRICE + price_add
    price     = round(base * phase["price_mult"] * (0.75 if is_promo else 1.0), 2)
 
    # Purchase timestamp — season holders buy weeks in advance; walk-ups same day
    days_before = random.randint(0, 2) if is_new_jersey_acq or is_lapsed_reactivation \
                  else random.randint(0, 21)
    purchase_ts = datetime.combine(game.game_date, datetime.min.time()) \
                  - timedelta(days=days_before) \
                  + timedelta(hours=random.randint(8, 20))
 
    return Row(
        ticket_id               = str(uuid.uuid4()),
        customer_id             = cid,
        game_id                 = game.game_id,
        game_date               = game.game_date,
        section_tier            = tier,
        seat_row                = random.randint(1, 30),
        seat_number             = random.randint(1, 24),
        ticket_price            = price,
        purchase_channel        = random.choice(["online", "box_office", "mobile_app", "reseller"]),
        purchase_ts             = purchase_ts,
        is_promo_ticket         = is_promo,
        is_jersey_night_game    = is_jersey_night,
        is_lapsed_reactivation  = is_lapsed_reactivation,
        season_phase            = next(
            (k for k, v in PHASE_WEIGHTS.items() if game.game_number in v["games"]), "bridge"
        ),
        _ingested_at            = datetime.now(),
        _source                 = "seed_generator_v1",
    )
 
# ── 1. Season holders: buy ALL home games upfront ─────────────────────────
holder_customers = [c for c in customer_list if c.is_season_holder]
for holder in holder_customers:
    for game in home_events:
        phase = get_phase(game.game_number)
        tickets.append(make_ticket(holder.customer_id, game, phase))
 
print(f"   Season holder tickets: {len(tickets):,}")
holder_ticket_count = len(tickets)
 
# ── 2. General admission — phase-weighted per game ────────────────────────
non_holders  = [c for c in customer_list if not c.is_season_holder]
casual_fans  = [c for c in non_holders if c.initial_segment == "casual_fan"]
promo_fans   = [c for c in non_holders if c.initial_segment == "promo_hunter"]
new_fans     = [c for c in non_holders if c.initial_segment == "high_value_new"]
lapsed_fans  = [c for c in non_holders if c.initial_segment in ("lapsed", "deeply_lapsed")]
 
for game in home_events:
    gn    = game.game_number
    phase = get_phase(gn)
    fill_min, fill_max = phase["fill_rate"]
    target_fill = random.uniform(fill_min, fill_max)
    already_sold = len(holder_customers)   # season holders fill some seats
    target_tickets = max(0, int(ARENA_CAPACITY * target_fill) - already_sold)
 
    # Build pool of eligible buyers for this game
    # Casual fans disappear during slump, promo fans require promos
    if gn in PHASE_WEIGHTS["slump"]["games"]:
        # Slump: fewer casual buyers, more promo-driven
        pool = (
            random.sample(casual_fans,  min(30, len(casual_fans)))  +
            random.sample(promo_fans,   min(120, len(promo_fans)))  +
            random.sample(new_fans,     min(20, len(new_fans)))
        )
    elif gn == JERSEY_NIGHT_GAME:
        # Jersey Night: everyone shows up including deeply lapsed
        pool = (
            random.sample(casual_fans,  min(200, len(casual_fans)))  +
            random.sample(promo_fans,   min(80,  len(promo_fans)))   +
            random.sample(new_fans,     min(150, len(new_fans)))     +
            random.sample(lapsed_fans,  min(94,  len(lapsed_fans)))  # The 94 deeply lapsed
        )
    elif gn in PHASE_WEIGHTS["hot_start"]["games"]:
        pool = (
            random.sample(casual_fans,  min(180, len(casual_fans))) +
            random.sample(new_fans,     min(120, len(new_fans)))    +
            random.sample(promo_fans,   min(40,  len(promo_fans)))
        )
    else:
        # Late push / post-jersey: recovering but not peak
        pool = (
            random.sample(casual_fans,  min(120, len(casual_fans))) +
            random.sample(new_fans,     min(80,  len(new_fans)))    +
            random.sample(promo_fans,   min(60,  len(promo_fans)))
        )
 
    random.shuffle(pool)
    buyers = pool[:target_tickets]
 
    for buyer in buyers:
        is_promo = random.random() < phase["promo_rate"]
        is_lapsed_reactivation = (
            gn == JERSEY_NIGHT_GAME and
            buyer.initial_segment in ("lapsed", "deeply_lapsed")
        )
        tickets.append(make_ticket(
            buyer.customer_id, game, phase,
            is_promo=is_promo,
            is_jersey_night=(gn == JERSEY_NIGHT_GAME),
            is_lapsed_reactivation=is_lapsed_reactivation,
        ))
 
print(f"   Total tickets (pre-Jersey Night new acq): {len(tickets):,}")
 
# ── 3. Jersey Night: 412 brand-new fans (acquisition spike) ──────────────
# These customers don't exist yet in raw_customers — we inject them
# to simulate the walk-up / event-driven signup surge.
# The simulator notebook (Phase 3) will drip these in dynamically,
# but we seed them here for the historical baseline.
 
jersey_new_customers = []
jersey_new_tickets   = []
jersey_game          = next(e for e in event_list if e.is_jersey_night)
jersey_phase         = get_phase(JERSEY_NIGHT_GAME)
 
for _ in range(412):
    cid   = str(uuid.uuid4())
    first = fake.first_name()
    last  = fake.last_name()
    jersey_new_customers.append(Row(
        customer_id          = cid,
        full_name            = f"{first} {last}",
        email                = f"{first.lower()}.{last.lower()}@{fake.free_email_domain()}",
        phone                = fake.phone_number(),
        zip_code             = random.choice(IDAHO_ZIPS),
        state                = "ID",
        acquisition_channel  = "jersey_night_event",
        initial_segment      = "high_value_new",
        is_season_holder     = False,
        signup_date          = jersey_game.game_date,
        is_jersey_night_acq  = True,
        is_deeply_lapsed     = False,
        _ingested_at         = datetime.now(),
        _source              = "seed_generator_v1",
    ))
    jersey_new_tickets.append(make_ticket(
        cid, jersey_game, jersey_phase,
        is_promo=False,
        is_jersey_night=True,
        is_new_jersey_acq=True,
    ))
 
# Append jersey night new fans to customers table
jn_customers_df = spark.createDataFrame(jersey_new_customers)
jn_customers_df.write.format("delta").mode("append") \
    .saveAsTable(f"{CATALOG}.bronze.raw_customers")
 
tickets.extend(jersey_new_tickets)
 
# ── Write all tickets ──────────────────────────────────────────────────────
tickets_df = spark.createDataFrame(tickets)
tickets_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.bronze.raw_tickets")
 
total_revenue = sum(t.ticket_price for t in tickets)
print(f"\n✅ raw_tickets written | {tickets_df.count():,} total tickets")
print(f"   Season holder tickets : {holder_ticket_count:,}")
print(f"   GA tickets            : {len(tickets) - holder_ticket_count - 412:,}")
print(f"   Jersey Night new acq  : 412")
print(f"   Total promo tickets   : {sum(1 for t in tickets if t.is_promo_ticket):,} "
      f"({sum(1 for t in tickets if t.is_promo_ticket)/len(tickets):.1%})")
print(f"   Simulated revenue     : ${total_revenue:,.2f}")

In [0]:
# COMMAND ----------
# ── CELL 8: Generate raw_promotions ──────────────────────────────────────
#
# BUSINESS CONTEXT:
#   Promos are the "bad story" engine. During the slump, marketing
#   over-indexes on discounts. We track promo type, discount depth,
#   and whether it was redeemed — so the Gold layer can calculate
#   true promo ROI and margin impact.
 
PROMO_TYPES = {
    "pct_discount":   0.45,   # % off face value (most common)
    "bogo":           0.20,   # Buy one get one
    "bundle_food":    0.15,   # Ticket + $15 food credit
    "group_rate":     0.12,   # 10+ group discount
    "loyalty_reward": 0.08,   # Season holder loyalty perk
}
promo_type_names    = list(PROMO_TYPES.keys())
promo_type_weights  = list(PROMO_TYPES.values())
 
DISCOUNT_BY_TYPE = {
    "pct_discount":   (0.15, 0.35),
    "bogo":           (0.50, 0.50),
    "bundle_food":    (0.10, 0.18),
    "group_rate":     (0.20, 0.30),
    "loyalty_reward": (0.10, 0.25),
}
 
promo_tickets = [t for t in tickets if t.is_promo_ticket]
promotions    = []
 
for ticket in promo_tickets:
    promo_type  = random.choices(promo_type_names, weights=promo_type_weights)[0]
    disc_min, disc_max = DISCOUNT_BY_TYPE[promo_type]
    discount_pct = round(random.uniform(disc_min, disc_max), 2)
 
    promotions.append(Row(
        promo_id        = str(uuid.uuid4()),
        ticket_id       = ticket.ticket_id,
        customer_id     = ticket.customer_id,
        game_id         = ticket.game_id,
        promo_type      = promo_type,
        discount_pct    = discount_pct,
        face_value      = round(ticket.ticket_price / (1 - discount_pct), 2),
        discount_amount = round(ticket.ticket_price / (1 - discount_pct) * discount_pct, 2),
        redeemed        = True,   # All promos in seed data were redeemed
        season_phase    = ticket.season_phase,
        _ingested_at    = datetime.now(),
        _source         = "seed_generator_v1",
    ))
 
promos_df = spark.createDataFrame(promotions)
promos_df.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{CATALOG}.bronze.raw_promotions")
 
total_discount = sum(p.discount_amount for p in promotions)
print(f"✅ raw_promotions written | {promos_df.count():,} promos")
print(f"   Total discount given  : ${total_discount:,.2f}")
print(f"   Avg discount per promo: ${total_discount/len(promotions):.2f}")
promo_by_phase = {}
for p in promotions:
    promo_by_phase[p.season_phase] = promo_by_phase.get(p.season_phase, 0) + 1
for phase, cnt in sorted(promo_by_phase.items(), key=lambda x: -x[1]):
    print(f"   {phase:<20}: {cnt:>5} promos")

In [0]:
# COMMAND ----------
# ── CELL 9: Validation Checks ─────────────────────────────────────────────
#
# Run these before moving to Phase 2.
# All checks should PASS. If any FAIL, investigate before proceeding.
 
print("=" * 60)
print("ICEBASE PHASE 1 — VALIDATION REPORT")
print("=" * 60)
 
checks = []
 
# Check 1: Table existence
tables = [r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOG}.bronze").collect()]
for tbl in ["raw_events", "raw_customers", "raw_tickets", "raw_promotions"]:
    status = "PASS" if tbl in tables else "FAIL"
    checks.append((status, f"Table {CATALOG}.bronze.{tbl} exists"))
 
# Check 2: Row counts in expected ranges
counts = {
    "raw_events":     spark.table(f"{CATALOG}.bronze.raw_events").count(),
    "raw_customers":  spark.table(f"{CATALOG}.bronze.raw_customers").count(),
    "raw_tickets":    spark.table(f"{CATALOG}.bronze.raw_tickets").count(),
    "raw_promotions": spark.table(f"{CATALOG}.bronze.raw_promotions").count(),
}
checks.append(("PASS" if counts["raw_events"] == TOTAL_GAMES else "FAIL",
    f"raw_events has exactly {TOTAL_GAMES} games (got {counts['raw_events']})"))
checks.append(("PASS" if counts["raw_customers"] >= N_CUSTOMERS_BASE else "FAIL",
    f"raw_customers >= {N_CUSTOMERS_BASE:,} (got {counts['raw_customers']:,})"))
checks.append(("PASS" if counts["raw_tickets"] > 10_000 else "FAIL",
    f"raw_tickets > 10,000 rows (got {counts['raw_tickets']:,})"))
checks.append(("PASS" if counts["raw_promotions"] > 500 else "FAIL",
    f"raw_promotions > 500 rows (got {counts['raw_promotions']:,})"))
 
# Check 3: Jersey Night exists and is a sellout
jn = spark.sql(f"""
    SELECT arena_fill_pct, is_jersey_night
    FROM {CATALOG}.bronze.raw_events
    WHERE is_jersey_night = true
""").collect()
checks.append(("PASS" if len(jn) == 1 else "FAIL",
    f"Exactly 1 Jersey Night game exists (got {len(jn)})"))
checks.append(("PASS" if jn and jn[0].arena_fill_pct == 1.0 else "FAIL",
    f"Jersey Night is a sellout (fill_pct = {jn[0].arena_fill_pct if jn else 'N/A'})"))
 
# Check 4: Jersey Night acquisition fans exist
jn_acq = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM {CATALOG}.bronze.raw_customers
    WHERE acquisition_channel = 'jersey_night_event'
""").collect()[0].cnt
checks.append(("PASS" if jn_acq == 412 else "FAIL",
    f"412 Jersey Night new acquisitions (got {jn_acq})"))
 
# Check 5: Promo rate during slump is higher than hot start
slump_promo = spark.sql(f"""
    SELECT
      SUM(CASE WHEN is_promo_ticket THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS promo_rate
    FROM {CATALOG}.bronze.raw_tickets
    WHERE season_phase = 'slump'
""").collect()[0].promo_rate
 
hot_promo = spark.sql(f"""
    SELECT
      SUM(CASE WHEN is_promo_ticket THEN 1 ELSE 0 END) * 1.0 / COUNT(*) AS promo_rate
    FROM {CATALOG}.bronze.raw_tickets
    WHERE season_phase = 'hot_start'
""").collect()[0].promo_rate
 
checks.append(("PASS" if slump_promo > hot_promo else "FAIL",
    f"Slump promo rate ({slump_promo:.1%}) > Hot Start promo rate ({hot_promo:.1%})"))
 
# Check 6: Dirty data exists in raw_customers (for Silver to clean)
dirty = spark.sql(f"""
    SELECT COUNT(*) AS cnt
    FROM {CATALOG}.bronze.raw_customers
    WHERE full_name != TRIM(full_name) OR full_name = UPPER(full_name)
""").collect()[0].cnt
checks.append(("PASS" if dirty > 0 else "WARN",
    f"Dirty name records exist for Silver cleaning ({dirty} found)"))
 
# ── Print report ──────────────────────────────────────────────────────────
print()
for status, msg in checks:
    icon = "✅" if status == "PASS" else ("⚠️ " if status == "WARN" else "❌")
    print(f"  {icon} [{status}] {msg}")
 
fails = sum(1 for s, _ in checks if s == "FAIL")
print()
if fails == 0:
    print("🏒 ALL CHECKS PASSED — Phase 1 complete. Ready for Phase 2 (DLT Pipeline).")
else:
    print(f"❌ {fails} check(s) FAILED — review before proceeding to Phase 2.")

In [0]:
# COMMAND ----------
# ── CELL 10: Quick Business Sanity Check ──────────────────────────────────
#
# These queries answer the business questions your CMO persona would ask.
# Run these to confirm the narrative is visible in raw data before
# building the pipeline on top of it.
 
print("=" * 60)
print("NARRATIVE SANITY CHECK — Key Business Metrics by Phase")
print("=" * 60)
 
spark.sql(f"""
    SELECT
        e.season_phase,
        COUNT(DISTINCT t.game_id)                               AS games,
        ROUND(AVG(e.arena_fill_pct) * 100, 1)                  AS avg_fill_pct,
        ROUND(AVG(t.ticket_price), 2)                          AS avg_ticket_price,
        ROUND(SUM(CASE WHEN t.is_promo_ticket THEN 1 ELSE 0 END)
              * 100.0 / COUNT(*), 1)                            AS promo_rate_pct,
        COUNT(DISTINCT t.customer_id)                          AS unique_buyers,
        ROUND(SUM(t.ticket_price), 2)                          AS total_revenue
    FROM {CATALOG}.bronze.raw_tickets t
    JOIN {CATALOG}.bronze.raw_events  e ON t.game_id = e.game_id
    GROUP BY e.season_phase
    ORDER BY MIN(e.game_number)
""").show(truncate=False)
 
print("\n📊 Jersey Night Snapshot:")
spark.sql(f"""
    SELECT
        COUNT(*)                                                AS total_tickets,
        SUM(CASE WHEN t.is_lapsed_reactivation THEN 1 ELSE 0 END) AS lapsed_reactivations,
        COUNT(DISTINCT CASE WHEN c.acquisition_channel = 'jersey_night_event'
                            THEN c.customer_id END)            AS new_fan_acquisitions,
        ROUND(AVG(t.ticket_price), 2)                          AS avg_ticket_price
    FROM {CATALOG}.bronze.raw_tickets t
    JOIN {CATALOG}.bronze.raw_customers c ON t.customer_id = c.customer_id
    WHERE t.is_jersey_night_game = true
""").show(truncate=False)